# Assignment Beta - Question 1

**Name:** Muhammad Sabeeh  
**Roll number:** 23K-0002  
**Course:** Agentic AI (AI4015)  
**Semester:** Fall 2026


# Question 1 calculations and evidence
All code cells below were executed locally. Muhammad Sabeeh | 23K-0002. Source register and raw logs accompany this notebook. Reported model profiles are separate from own-machine tool timings.

In [1]:
import math,json,csv,statistics
from pathlib import Path
from datetime import datetime,timezone
ROLL="23K-0002"
ROOT=next(p for p in [Path.cwd(),Path.cwd()/"q1",Path.cwd()/"assignment_beta"] if (p/"evidence/timing_summary.json").exists())

def q1():
    digits=[int(c) for c in ROLL if c.isdigit()]; R=10*digits[-2]+digits[-1]; D=sum(digits)%10
    Q=1500+25*R; N=5+R%4; p=round(.04+.01*D,2); H=4+R%4; B=1200+10*R; T=10-R%3
    steps=[{'k':k,'input':2000+800*(k-1),'output':600 if k==N else 350,
            'attempts_expected':1 if k==N else 1+p} for k in range(1,N+1)]
    I=round(sum(s['input']*s['attempts_expected'] for s in steps),6)
    O=round(sum(s['output']*s['attempts_expected'] for s in steps),6)
    C=round(sum(s['attempts_expected'] for s in steps),6); M=Q*(30+4*H)
    peak_qpm=5*Q*.35/120
    cfgpath=ROOT/'sources'/'qwen3_32b_config.json'
    cfg=json.loads(cfgpath.read_text()) if cfgpath.exists() else {'num_hidden_layers':64,'num_key_value_heads':8,'hidden_size':5120,'num_attention_heads':64,'head_dim':128}
    layers=cfg['num_hidden_layers']; kv=cfg['num_key_value_heads']; dim=cfg.get('head_dim',cfg['hidden_size']//cfg['num_attention_heads'])
    params=32762123264  # BF16 safetensors metadata 65,524,246,528 bytes / 2.
    index=ROOT/'sources'/'qwen3_32b_metadata.json'
    if index.exists():
        data=json.loads(index.read_text())
        params=data['metadata']['total_size']//2
    seq=steps[-1]['input']+steps[-1]['output']
    cache_bytes=2*layers*kv*dim*seq*2
    # A100 80GB treated conservatively as 80e9 bytes, not 80GiB.
    capacity=math.floor((.9*80e9-params/2)/cache_bytes)
    baseline_latency_seconds=60.0  # Capacity scenario; never a measured serving latency.
    sessions=math.ceil(peak_qpm*baseline_latency_seconds/60)
    gpus=math.ceil(sessions/capacity)
    return {'student_roll':ROLL,'student_name':'Muhammad Sabeeh','R':R,'D':D,'Q':Q,'N':N,'p':p,'H':H,'B':B,'T':T,'steps':steps,
            'input_per_query':I,'output_per_query':O,'calls_per_query':C,'monthly_queries':M,
            'monthly_input':M*I,'monthly_output':M*O,'monthly_calls':M*C,
            'prefix_per_query':1500*C,'peak_queries_per_minute':peak_qpm,
            'peak_calls_per_minute':peak_qpm*C,'peak_input_tpm':peak_qpm*I,
            'peak_output_tpm':peak_qpm*O,'peak_total_tpm':peak_qpm*(I+O),
            'memory':{'parameter_count':params,'layers':layers,'kv_heads':kv,'head_dim':dim,'sequence_length':seq,
                      'fp16_weight_bytes':params*2,'int8_weight_bytes':params,'int4_weight_bytes':params/2,
                      'kv_bytes_per_sequence':cache_bytes,'gpu_bytes':80e9,'concurrent_sequences_per_gpu':capacity,
                      'assumed_session_seconds':baseline_latency_seconds,'peak_sessions':sessions,'gpu_count':gpus,
                      'gpu_usd_per_hour':1.59,'monthly_usd':gpus*720*1.59}}

x=q1()
print(json.dumps(x,indent=2))

{
  "student_roll": "23K-0002",
  "student_name": "Muhammad Sabeeh",
  "R": 2,
  "D": 7,
  "Q": 1550,
  "N": 7,
  "p": 0.11,
  "H": 6,
  "B": 1220,
  "T": 8,
  "steps": [
    {
      "k": 1,
      "input": 2000,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 2,
      "input": 2800,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 3,
      "input": 3600,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 4,
      "input": 4400,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 5,
      "input": 5200,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 6,
      "input": 6000,
      "output": 350,
      "attempts_expected": 1.11
    },
    {
      "k": 7,
      "input": 6800,
      "output": 600,
      "attempts_expected": 1
    }
  ],
  "input_per_query": 33440.0,
  "output_per_query": 2931.0,
  "calls_per_query": 7.66,
  "monthly_queries": 83700,
  "mon

## Twelve routes and cache costs
Six closed and six open models, four developers per side. Azure GPT-4.1 routes remain publicly listed; Sonnet is close to retirement. Unknown entitlement/cache eligibility is disclosed, never entered as zero. Warm-cache scenario assumes 720 prefix creations; Flash-Lite 1500 tokens is below the externally observed 2048 minimum. Routes without verified discounts receive none.

In [2]:
MODELS=[{'model': 'GPT-4.1 mini', 'side': 'closed', 'developer': 'OpenAI', 'parameters': 'not disclosed', 'context_tokens': 1047576, 'max_output_tokens': 32768, 'input_usd_per_million': 0.39999999999999997, 'output_usd_per_million': 1.5999999999999999, 'cache_read_usd_per_million': 0.09999999999999999, 'cache_minimum_tokens': 1024, 'cache_ttl': 'In-memory: typically 5-10 min inactivity; removed within 1 h of last use', 'cache_write_multiplier': 1, 'rate_provider': 'Azure / azure via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 1.64, 'output_tokens_per_second': 78, 'tool_benchmark': 'BFCL v4 FC: 50.45% [B1]', 'general_benchmark': 'MMLU: 87.5%; OpenAI launch table [B2]', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'Non-reasoning; no reasoning-effort mode', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/openai/gpt-4.1-mini/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'model_id': 'openai/gpt-4.1-mini', 'speed_source': 'https://artificialanalysis.ai/models/gpt-4-1-mini', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['temperature', 'top_p', 'max_completion_tokens', 'tools', 'tool_choice', 'structured_outputs', 'response_format', 'seed'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Azure route active; OpenAI-direct nano snapshot shows shut down. Availability claim is for this quoted Azure route, not OpenAI direct.', 'cache_policy_source': 'https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/prompt-caching', 'field_sources': '{"benchmarks": "https://openai.com/index/gpt-4-1/", "cache": "https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/prompt-caching", "function_benchmark": "https://gorilla.cs.berkeley.edu/leaderboard.html", "parameters_license_reasoning": "https://openrouter.ai/api/v1/models/openai/gpt-4.1-mini/endpoints", "price_context_output_features": "https://openrouter.ai/api/v1/models/openai/gpt-4.1-mini/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/gpt-4-1-mini"}'}, {'model': 'GPT-4.1 nano', 'side': 'closed', 'developer': 'OpenAI', 'parameters': 'not disclosed', 'context_tokens': 1047576, 'max_output_tokens': 32768, 'input_usd_per_million': 0.09999999999999999, 'output_usd_per_million': 0.39999999999999997, 'cache_read_usd_per_million': 0.03, 'cache_minimum_tokens': 1024, 'cache_ttl': 'In-memory: typically 5-10 min inactivity; removed within 1 h of last use', 'cache_write_multiplier': 1, 'rate_provider': 'Azure / azure via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 0.99, 'output_tokens_per_second': 161, 'tool_benchmark': 'BFCL v4 FC (2025-04-14): 33.05% [B1]', 'general_benchmark': 'MMLU: 80.1% [B2]', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'Non-reasoning; no reasoning-effort mode', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/openai/gpt-4.1-nano/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'speed_source': 'https://artificialanalysis.ai/models/gpt-4-1-nano', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'model_id': 'openai/gpt-4.1-nano', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['temperature', 'top_p', 'max_completion_tokens', 'tool_choice', 'tools', 'structured_outputs', 'response_format', 'seed'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Azure route active; OpenAI-direct nano snapshot shows shut down. Availability claim is for this quoted Azure route, not OpenAI direct.', 'cache_policy_source': 'https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/prompt-caching', 'field_sources': '{"benchmarks": "https://openai.com/index/gpt-4-1/", "cache": "https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/prompt-caching", "function_benchmark": "https://gorilla.cs.berkeley.edu/leaderboard.html", "parameters_license_reasoning": "https://openrouter.ai/api/v1/models/openai/gpt-4.1-nano/endpoints", "price_context_output_features": "https://openrouter.ai/api/v1/models/openai/gpt-4.1-nano/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/gpt-4-1-nano"}'}, {'model': 'Claude Haiku 4.5', 'side': 'closed', 'developer': 'Anthropic', 'parameters': 'not disclosed', 'context_tokens': 200000, 'max_output_tokens': 64000, 'input_usd_per_million': 1.0, 'output_usd_per_million': 5.0, 'cache_read_usd_per_million': 0.09999999999999999, 'cache_minimum_tokens': 4096, 'cache_ttl': '5 min default; 1 h optional; cache write multipliers 1.25x / 2x', 'cache_write_multiplier': 1.25, 'rate_provider': 'Anthropic / anthropic via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 1.17, 'output_tokens_per_second': 93, 'tool_benchmark': 'BFCL v4 FC: 68.70% [B1]', 'general_benchmark': 'SWE-bench Verified: 73.3%; 50 trials, 128K thinking budget; not non-thinking evidence', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking disabled; optional extended thinking', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/anthropic/claude-haiku-4.5/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'cache_policy_source': 'https://platform.claude.com/docs/en/build-with-claude/prompt-caching', 'benchmark_source': 'https://www.anthropic.com/news/claude-haiku-4-5', 'model_id': 'anthropic/claude-haiku-4.5', 'speed_source': 'https://artificialanalysis.ai/models/claude-4-5-haiku', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['reasoning', 'include_reasoning', 'max_tokens', 'temperature', 'top_p', 'top_k', 'stop', 'tools', 'tool_choice', 'structured_outputs', 'response_format'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'field_sources': '{"benchmarks": "https://www.anthropic.com/news/claude-haiku-4-5", "cache": "https://platform.claude.com/docs/en/build-with-claude/prompt-caching", "function_benchmark": "https://gorilla.cs.berkeley.edu/leaderboard.html", "parameters_license_reasoning": "https://www.anthropic.com/news/claude-haiku-4-5", "price_context_output_features": "https://openrouter.ai/api/v1/models/anthropic/claude-haiku-4.5/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/claude-4-5-haiku"}'}, {'model': 'Claude Sonnet 4.5', 'side': 'closed', 'developer': 'Anthropic', 'parameters': 'not disclosed', 'context_tokens': 1000000, 'max_output_tokens': 64000, 'input_usd_per_million': 3.0, 'output_usd_per_million': 15.0, 'cache_read_usd_per_million': 0.3, 'cache_minimum_tokens': 1024, 'cache_ttl': '5 min default; 1 h optional; cache write multipliers 1.25x / 2x', 'cache_write_multiplier': 1.25, 'rate_provider': 'Anthropic / anthropic via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 1.29, 'output_tokens_per_second': 59, 'tool_benchmark': 'BFCL v4 FC (20250929): 73.24% [B1]', 'general_benchmark': 'SWE-bench Verified: 77.2%; 10 trials, 200K thinking budget; not non-thinking evidence', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking disabled; optional extended thinking', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/anthropic/claude-sonnet-4.5/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'cache_policy_source': 'https://platform.claude.com/docs/en/build-with-claude/prompt-caching', 'benchmark_source': 'https://www.anthropic.com/news/claude-sonnet-4-5', 'model_id': 'anthropic/claude-sonnet-4.5', 'speed_source': 'https://artificialanalysis.ai/models/claude-4-5-sonnet', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['reasoning', 'include_reasoning', 'max_tokens', 'temperature', 'top_p', 'top_k', 'stop', 'tools', 'tool_choice', 'structured_outputs', 'response_format'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Listed serving route; Anthropic retirement scheduled 2026-11-30. Exclude from a new production contract.', 'field_sources': '{"benchmarks": "https://www.anthropic.com/news/claude-sonnet-4-5", "cache": "https://platform.claude.com/docs/en/build-with-claude/prompt-caching", "function_benchmark": "https://gorilla.cs.berkeley.edu/leaderboard.html", "parameters_license_reasoning": "https://www.anthropic.com/news/claude-sonnet-4-5", "price_context_output_features": "https://openrouter.ai/api/v1/models/anthropic/claude-sonnet-4.5/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/claude-4-5-sonnet"}'}, {'model': 'Gemini 2.5 Flash-Lite', 'side': 'closed', 'developer': 'Google', 'parameters': 'not disclosed', 'context_tokens': 1048576, 'max_output_tokens': 65535, 'input_usd_per_million': 0.09999999999999999, 'output_usd_per_million': 0.39999999999999997, 'cache_read_usd_per_million': 0.01, 'cache_minimum_tokens': 2048, 'cache_ttl': 'Explicit default 1 h; configurable; implicit best effort', 'cache_write_multiplier': 1, 'rate_provider': 'Google AI Studio / google-ai-studio via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 0.44, 'output_tokens_per_second': 307, 'tool_benchmark': 'BFCL v4 FC: 36.87% [B1]', 'general_benchmark': 'Global MMLU Lite: 81.1%, June 17 non-thinking release', 'native_tools': 'Yes', 'structured_json': 'Yes', 'reasoning_setting': 'thinking_budget=0; optional thinking budget', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/google/gemini-2.5-flash-lite/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'benchmark_source': 'https://storage.googleapis.com/deepmind-media/Model-Cards/Gemini-2-5-Flash-Lite-Model-Card.pdf', 'speed_source': 'https://artificialanalysis.ai/models/gemini-2-5-flash-lite', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'model_id': 'google/gemini-2.5-flash-lite', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['reasoning', 'include_reasoning', 'structured_outputs', 'response_format', 'max_tokens', 'temperature', 'top_p', 'seed', 'tools', 'tool_choice', 'stop'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://aistudio.google.com/docs/caching?generate-content=true', 'cache_minimum_source': 'https://discuss.ai.google.dev/t/context-caching-docs-list-wrong-minimum-token-count-for-gemini-2-5-flash-flash-lite-missing-entirely/178942', 'cache_minimum_caveat': '2048 reported from actual API rejection by external developer; current official table omits Flash-Lite. Conservatively no discount for 1500 tokens.', 'field_sources': '{"benchmarks": "https://storage.googleapis.com/deepmind-media/Model-Cards/Gemini-2-5-Flash-Lite-Model-Card.pdf", "cache": "https://aistudio.google.com/docs/caching?generate-content=true", "cache_minimum_external_observation": "https://discuss.ai.google.dev/t/context-caching-docs-list-wrong-minimum-token-count-for-gemini-2-5-flash-flash-lite-missing-entirely/178942", "function_benchmark": "https://gorilla.cs.berkeley.edu/leaderboard.html", "parameters_license_reasoning": "https://storage.googleapis.com/deepmind-media/Model-Cards/Gemini-2-5-Flash-Lite-Model-Card.pdf", "price_context_output_features": "https://openrouter.ai/api/v1/models/google/gemini-2.5-flash-lite/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/gemini-2-5-flash-lite"}'}, {'model': 'Amazon Nova Lite', 'side': 'closed', 'developer': 'Amazon', 'parameters': 'not disclosed', 'context_tokens': 300000, 'max_output_tokens': 5120, 'input_usd_per_million': 0.06, 'output_usd_per_million': 0.24, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': 1024, 'cache_ttl': 'Bedrock explicit: 5 min; no discounted cache price exposed on selected OpenRouter route', 'cache_write_multiplier': 1, 'rate_provider': 'Amazon Bedrock / amazon-bedrock/eu-west-1 via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 1.66, 'output_tokens_per_second': 124, 'tool_benchmark': 'BFCL v3 overall 66.6%; Nov 17 2024 benchmark snapshot, report Table 4', 'general_benchmark': 'MMLU 80.5%; 0-shot CoT prompt, report Table 1; CoT is visible output, not hidden reasoning tokens', 'native_tools': 'Yes; Converse tool specification', 'structured_json': 'Prompted JSON; no strict schema guarantee on selected route', 'reasoning_setting': 'Non-reasoning; no separate hidden-reasoning mode', 'license': 'Proprietary', 'commercial_self_host': 'No', 'source_url': 'https://openrouter.ai/api/v1/models/amazon/nova-lite-v1/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'model_id': 'amazon/nova-lite-v1', 'speed_source': 'https://artificialanalysis.ai/models/nova-lite', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'endpoint_supported_parameters': ['max_tokens', 'temperature', 'top_p', 'top_k', 'stop', 'tools'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'benchmark_source': 'https://cdn.amazon.science/96/7d/0d3e59514abf8fdcfafcdc574300/nova-tech-report-20250317-0810.pdf', 'cache_policy_source': 'https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html', 'cache_route_note': 'Native Bedrock explicit caching exists; this routed quote has no cached-input SKU, so no discount is assumed.', 'field_sources': '{"benchmarks": "https://cdn.amazon.science/96/7d/0d3e59514abf8fdcfafcdc574300/nova-tech-report-20250317-0810.pdf", "cache": "https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html", "parameters_license_reasoning": "https://cdn.amazon.science/96/7d/0d3e59514abf8fdcfafcdc574300/nova-tech-report-20250317-0810.pdf", "price_context_output_features": "https://openrouter.ai/api/v1/models/amazon/nova-lite-v1/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/nova-lite"}'}, {'model': 'Qwen3-235B-A22B-Instruct-2507', 'side': 'open', 'developer': 'Alibaba Qwen', 'parameters': '235B total / 22B active MoE', 'context_tokens': 262144, 'max_output_tokens': 235929, 'input_usd_per_million': 0.072, 'output_usd_per_million': 0.44, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'Not applicable to costing: selected route exposes no discounted cache SKU', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 3.96, 'output_tokens_per_second': 47, 'tool_benchmark': 'BFCL v3: 70.9% [B5]', 'general_benchmark': 'MMLU-Redux: 93.1% [B5]', 'native_tools': 'Checkpoint supports tools; selected endpoint advertises tools', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'Non-thinking instruct', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/qwen/qwen3-235b-a22b-2507/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'model_id': 'qwen/qwen3-235b-a22b-2507', 'speed_source': 'https://artificialanalysis.ai/models/qwen3-235b-a22b-instruct-2507', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['max_tokens', 'temperature', 'top_p', 'stop', 'frequency_penalty', 'presence_penalty', 'repetition_penalty', 'top_k', 'seed', 'min_p', 'logit_bias', 'response_format', 'structured_outputs', 'tools', 'tool_choice'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://docs.deepinfra.com/chat/prompt-caching', 'field_sources': '{"benchmarks": "https://gorilla.cs.berkeley.edu/leaderboard.html", "cache": "https://docs.deepinfra.com/chat/prompt-caching", "parameters_license_reasoning": "https://huggingface.co/Qwen/Qwen3-235B-A22B-Instruct-2507", "price_context_output_features": "https://openrouter.ai/api/v1/models/qwen/qwen3-235b-a22b-2507/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/qwen3-235b-a22b-instruct-2507"}'}, {'model': 'Qwen3-32B', 'side': 'open', 'developer': 'Alibaba Qwen', 'parameters': '32.762B dense', 'context_tokens': 40960, 'max_output_tokens': 16384, 'input_usd_per_million': 0.064, 'output_usd_per_million': 0.224, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'Not applicable to costing: selected route exposes no discounted cache SKU', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 1.09, 'output_tokens_per_second': 103, 'tool_benchmark': 'BFCL v3: 63.0%, non-thinking, Table 14', 'general_benchmark': 'MMLU-Redux: 85.7%, non-thinking, Table 14', 'native_tools': 'Checkpoint supports tools; selected endpoint does not advertise tools; do not promise tool API support on this cheapest route', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'enable_thinking=False', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/qwen/qwen3-32b/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'benchmark_source': 'https://arxiv.org/html/2505.09388v1', 'model_id': 'qwen/qwen3-32b', 'speed_source': 'https://artificialanalysis.ai/models/qwen3-32b-instruct', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['reasoning', 'include_reasoning', 'max_tokens', 'temperature', 'top_p', 'stop', 'frequency_penalty', 'presence_penalty', 'repetition_penalty', 'top_k', 'seed', 'min_p', 'logit_bias', 'response_format', 'structured_outputs'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://docs.deepinfra.com/chat/prompt-caching', 'field_sources': '{"benchmarks": "https://arxiv.org/html/2505.09388v1", "cache": "https://docs.deepinfra.com/chat/prompt-caching", "parameters_license_reasoning": "https://huggingface.co/Qwen/Qwen3-32B", "price_context_output_features": "https://openrouter.ai/api/v1/models/qwen/qwen3-32b/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/qwen3-32b-instruct"}'}, {'model': 'Llama-3.3-70B-Instruct', 'side': 'open', 'developer': 'Meta', 'parameters': '70B dense', 'context_tokens': 131072, 'max_output_tokens': 16384, 'input_usd_per_million': 0.09999999999999999, 'output_usd_per_million': 0.32, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'Not applicable to costing: selected route exposes no discounted cache SKU', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/turbo via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 2.23, 'output_tokens_per_second': 64, 'tool_benchmark': 'BFCL v2: 77.3% [B3]', 'general_benchmark': 'MMLU Pro CoT 5-shot: 68.9% [B3]', 'native_tools': 'Checkpoint supports tools; selected endpoint advertises tools', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'Non-reasoning instruct', 'license': 'Llama 3.3 Community', 'commercial_self_host': 'Conditional; review licence', 'source_url': 'https://openrouter.ai/api/v1/models/meta-llama/llama-3.3-70b-instruct/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'model_id': 'meta-llama/llama-3.3-70b-instruct', 'speed_source': 'https://artificialanalysis.ai/models/llama-3-3-instruct-70b', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['max_tokens', 'temperature', 'top_p', 'stop', 'frequency_penalty', 'presence_penalty', 'repetition_penalty', 'top_k', 'seed', 'min_p', 'tools', 'tool_choice', 'logit_bias', 'response_format', 'structured_outputs'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://docs.deepinfra.com/chat/prompt-caching', 'field_sources': '{"benchmarks": "https://gorilla.cs.berkeley.edu/leaderboard.html", "cache": "https://docs.deepinfra.com/chat/prompt-caching", "parameters_license_reasoning": "https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct", "price_context_output_features": "https://openrouter.ai/api/v1/models/meta-llama/llama-3.3-70b-instruct/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/llama-3-3-instruct-70b"}'}, {'model': 'Mistral-Small-3.2-24B-Instruct', 'side': 'open', 'developer': 'Mistral AI', 'parameters': '24B dense', 'context_tokens': 128000, 'max_output_tokens': 102400, 'input_usd_per_million': 0.06, 'output_usd_per_million': 0.16, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'Not applicable to costing: selected route exposes no discounted cache SKU', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp8/flex via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 2.57, 'output_tokens_per_second': 33, 'tool_benchmark': 'BFCL v4 FC (Mistral-small-2506): 37.15% [B1]', 'general_benchmark': 'MMLU: 80.50% [B4]', 'native_tools': 'Checkpoint supports tools; selected endpoint advertises tools', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'Non-reasoning instruct', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/mistralai/mistral-small-3.2-24b-instruct/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'model_id': 'mistralai/mistral-small-3.2-24b-instruct', 'speed_source': 'https://artificialanalysis.ai/models/mistral-small-3-2', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['max_tokens', 'temperature', 'top_p', 'stop', 'frequency_penalty', 'presence_penalty', 'repetition_penalty', 'top_k', 'seed', 'min_p', 'logit_bias', 'response_format', 'structured_outputs', 'tools'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://docs.deepinfra.com/chat/prompt-caching', 'field_sources': '{"benchmarks": "https://gorilla.cs.berkeley.edu/leaderboard.html", "cache": "https://docs.deepinfra.com/chat/prompt-caching", "parameters_license_reasoning": "https://huggingface.co/mistralai/Mistral-Small-3.2-24B-Instruct-2506", "price_context_output_features": "https://openrouter.ai/api/v1/models/mistralai/mistral-small-3.2-24b-instruct/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/mistral-small-3-2"}'}, {'model': 'DeepSeek-V3.1', 'side': 'open', 'developer': 'DeepSeek', 'parameters': '671B total / 37B active MoE; publisher card (AA displays a different 685B count)', 'context_tokens': 163840, 'max_output_tokens': 147456, 'input_usd_per_million': 0.25, 'output_usd_per_million': 0.95, 'cache_read_usd_per_million': 0.13, 'cache_minimum_tokens': None, 'cache_ttl': 'Automatic cache expires after inactivity; numeric TTL/granularity not published for this model/route. Direct paid retention offers 5m/1h at 1.25x/2x, not assumed routed.', 'cache_write_multiplier': 1, 'rate_provider': 'DeepInfra / deepinfra/fp4 via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 2.73, 'output_tokens_per_second': 70, 'tool_benchmark': 'Terminal-bench / Terminus 1: 31.3%, non-thinking', 'general_benchmark': 'MMLU-Redux: 91.8%, non-thinking', 'native_tools': 'Checkpoint supports tools; selected endpoint advertises tools', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'Non-thinking template', 'license': 'MIT', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/deepseek/deepseek-chat-v3.1/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'benchmark_source': 'https://huggingface.co/deepseek-ai/DeepSeek-V3.1/raw/main/README.md', 'model_id': 'deepseek/deepseek-chat-v3.1', 'speed_source': 'https://artificialanalysis.ai/models/deepseek-v3-1', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['reasoning', 'include_reasoning', 'max_tokens', 'temperature', 'top_p', 'stop', 'frequency_penalty', 'presence_penalty', 'repetition_penalty', 'top_k', 'seed', 'min_p', 'tools', 'tool_choice', 'structured_outputs', 'logit_bias', 'response_format'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://docs.deepinfra.com/chat/prompt-caching', 'field_sources': '{"benchmarks": "https://huggingface.co/deepseek-ai/DeepSeek-V3.1/raw/main/README.md", "cache": "https://docs.deepinfra.com/chat/prompt-caching", "parameters_license_reasoning": "https://huggingface.co/deepseek-ai/DeepSeek-V3.1", "price_context_output_features": "https://openrouter.ai/api/v1/models/deepseek/deepseek-chat-v3.1/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/deepseek-v3-1"}'}, {'model': 'Qwen3-30B-A3B-Instruct-2507', 'side': 'open', 'developer': 'Alibaba Qwen', 'parameters': '30.5B total / 3.3B active MoE', 'context_tokens': 262144, 'max_output_tokens': 235929, 'input_usd_per_million': 0.09, 'output_usd_per_million': 0.3, 'cache_read_usd_per_million': None, 'cache_minimum_tokens': None, 'cache_ttl': 'Not applicable to costing: selected route exposes no discounted cache SKU', 'cache_write_multiplier': 1, 'rate_provider': 'SiliconFlow / siliconflow/fp8 via OpenRouter', 'new_paid_rpm': 'No fixed platform request cap for paid model routes; provider/IP/spend constraints still apply', 'new_paid_tpm': 'No public fixed new-account TPM entitlement; account/provider dependent (NR)', 'ttft_seconds': 2.2, 'output_tokens_per_second': 59, 'tool_benchmark': 'BFCL v3 65.1%', 'general_benchmark': 'MMLU-Redux 89.3%', 'native_tools': 'Checkpoint supports tools; selected endpoint advertises tools', 'structured_json': 'Schema output advertised', 'reasoning_setting': 'Non-thinking instruct; no think blocks', 'license': 'Apache-2.0', 'commercial_self_host': 'Yes', 'source_url': 'https://openrouter.ai/api/v1/models/qwen/qwen3-30b-a3b-instruct-2507/endpoints', 'access_date': '2026-11-26', 'evidence_status': 'Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified', 'quota_source': 'https://openrouter.ai/docs/api/reference/limits', 'benchmark_source': 'https://huggingface.co/Qwen/Qwen3-30B-A3B-Instruct-2507', 'model_id': 'qwen/qwen3-30b-a3b-instruct-2507', 'speed_source': 'https://artificialanalysis.ai/models/qwen3-30b-a3b-2507', 'speed_caveat': 'AA Performance v3 END-TO-END output rate includes TTFT; selected non-reasoning row. Provider or provider-median profile, not a guarantee for cheapest route. Do not treat as post-TTFT decode speed.', 'speed_definition': 'end-to-end tokens / total request seconds', 'endpoint_supported_parameters': ['structured_outputs', 'response_format', 'temperature', 'top_p', 'top_k', 'frequency_penalty', 'tools', 'tool_choice', 'max_tokens'], 'extra_reasoning_tokens_per_call': 0, 'availability': 'Public listed serving endpoint exists; current means available, not latest release.', 'cache_policy_source': 'https://openrouter.ai/api/v1/models/qwen/qwen3-30b-a3b-instruct-2507/endpoints', 'field_sources': '{"benchmarks": "https://huggingface.co/Qwen/Qwen3-30B-A3B-Instruct-2507", "cache": "https://openrouter.ai/api/v1/models/qwen/qwen3-30b-a3b-instruct-2507/endpoints", "parameters_license_reasoning": "https://huggingface.co/Qwen/Qwen3-30B-A3B-Instruct-2507", "price_context_output_features": "https://openrouter.ai/api/v1/models/qwen/qwen3-30b-a3b-instruct-2507/endpoints", "quota": "https://openrouter.ai/docs/api/reference/limits", "speed": "https://artificialanalysis.ai/models/qwen3-30b-a3b-2507"}'}]
def model_cost(m,x):
    a=m['input_usd_per_million'];b=m['output_usd_per_million'];c=m['cache_read_usd_per_million']
    raw=(x['monthly_input']*a+x['monthly_output']*b)/1e6
    attempts=x['monthly_calls'];prefix=1500
    eligible=c is not None and m['cache_minimum_tokens'] is not None and prefix>=m['cache_minimum_tokens']
    # One cache creation per hour as a conservative explicit refresh policy.
    # All other matching calls hit the cache. Assumes traffic is regular enough
    # to keep the prefix warm; actual cache hit fraction must be logged.
    writes=720 if eligible else 0
    hits=max(0,attempts-writes) if eligible else 0
    write_multiplier=m['cache_write_multiplier']
    # For OpenAI only 1,408 of 1,500 identical tokens lie on a 128-token boundary.
    cached_prefix=1408 if m['developer']=='OpenAI' else prefix
    reduction=hits*cached_prefix*(a-(c or 0))/1e6
    premium=writes*cached_prefix*a*(write_multiplier-1)/1e6
    storage=1500*720/1e6 if eligible and m['developer']=='Google' else 0
    cached=raw-reduction+premium+storage
    reasoning_unknown=m['model'] in ['Grok 4.7','gpt-oss-120b']
    return {'model':m['model'],'rate_evidence':m['evidence_status'],'uncached_usd':raw,
            'cache_scenario_usd':cached,'eligible_prefix':eligible,'billable_cached_prefix':cached_prefix if eligible else 0,
            'cache_creations_assumed':writes,'cache_hits_assumed':hits,'write_premium_usd':premium,
            'storage_usd':storage,'within_budget_uncached':None if reasoning_unknown and raw<=x['B'] else raw<=x['B'],
            'within_budget_cache_scenario':None if reasoning_unknown and cached<=x['B'] else cached<=x['B'],
            'reasoning_surcharge_unknown':reasoning_unknown}

COSTS=[model_cost(m,x) for m in MODELS]
print(json.dumps(COSTS,indent=2))

[
  {
    "model": "GPT-4.1 mini",
    "rate_evidence": "Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified",
    "uncached_usd": 1512.09072,
    "cache_scenario_usd": 1241.5764672,
    "eligible_prefix": true,
    "billable_cached_prefix": 1408,
    "cache_creations_assumed": 720,
    "cache_hits_assumed": 640422.0,
    "write_premium_usd": 0.0,
    "storage_usd": 0,
    "within_budget_uncached": false,
    "within_budget_cache_scenario": false,
    "reasoning_surcharge_unknown": false
  },
  {
    "model": "GPT-4.1 nano",
    "rate_evidence": "Public route listed active on 2026-11-26; primary endpoint quote and AA non-reasoning speed profile; unknown entitlements/cache conditions explicitly qualified",
    "uncached_usd": 378.02268,
    "cache_scenario_usd": 314.90268768,
    "eligible_prefix": true,
    "billable_cached_prefix": 1408,
    "cache_creations_assumed": 720,
    "

## Own-machine timing evidence
Use the latest 20 successful calls per tool per date; preserve earlier attempts. Modelled air data is kept separate from observed station data. Dates are original Karachi timestamps.

In [3]:
rows=[]
for p in sorted((ROOT/"evidence").glob("tool_latency_primary_*.csv")):
    rows.extend(csv.DictReader(p.open(encoding="utf-8")))
assert len(rows)==160
assert {r["timestamp_karachi"][:10] for r in rows}=={"2026-11-25","2026-11-26"}
d=statistics.median(float(r["elapsed_seconds"]) for r in rows)
per={name:statistics.median(float(r["elapsed_seconds"]) for r in rows if r["tool"]==name) for name in ["weather","air_quality_observed","heat_index","knowledge_base"]}
print("Records:",len(rows),"Pooled median:",d,"Per tool:",per)

Records: 160 Pooled median: 0.5199098500306718 Per tool: {'weather': 1.6500752000138164, 'air_quality_observed': 1.858495699998457, 'heat_index': 2.1300045773386955e-05, 'knowledge_base': 2.3999949917197227e-05}


## Reported profiles and sequential latency
AA end-to-end output rate includes TTFT. Estimate each call as max(TTFT,output/rate), plus an explicitly assumed context-growth increment. Do not add TTFT twice. The profile is transferred from a benchmark workload, not measured at Karachi. Retry attempts remain in all terms.

In [4]:
def reported_latency(x,ttft,tps,delay,slope=.00005):
    terms=[]
    for step in x['steps']:
        extra=slope*(step['input']-2000)
        # Reported TPS includes TTFT already. Do not add TTFT a second time.
        seconds=max(ttft,step['output']/tps)+extra
        terms.append({**step,'reported_ttft_seconds':ttft,'reported_end_to_end_tps':tps,
                      'context_increment_assumed_seconds':extra,'llm_seconds_expected':seconds*step['attempts_expected']})
    return {'terms':terms,'seconds':sum(t['llm_seconds_expected'] for t in terms)+(x['N']-1)*delay,
            'tool_delay_measured':delay,'status':'ESTIMATE from reported profile and measured tools; context slope assumed'}

CANDIDATES=[m for m in MODELS if m["model"] in ["Gemini 2.5 Flash-Lite","Amazon Nova Lite"]]
for m in CANDIDATES:
    print(m["model"],json.dumps(reported_latency(x,m["ttft_seconds"],m["output_tokens_per_second"],d),indent=2))

Gemini 2.5 Flash-Lite {
  "terms": [
    {
      "k": 1,
      "input": 2000,
      "output": 350,
      "attempts_expected": 1.11,
      "reported_ttft_seconds": 0.44,
      "reported_end_to_end_tps": 307,
      "context_increment_assumed_seconds": 0.0,
      "llm_seconds_expected": 1.2654723127035832
    },
    {
      "k": 2,
      "input": 2800,
      "output": 350,
      "attempts_expected": 1.11,
      "reported_ttft_seconds": 0.44,
      "reported_end_to_end_tps": 307,
      "context_increment_assumed_seconds": 0.04,
      "llm_seconds_expected": 1.3098723127035832
    },
    {
      "k": 3,
      "input": 3600,
      "output": 350,
      "attempts_expected": 1.11,
      "reported_ttft_seconds": 0.44,
      "reported_end_to_end_tps": 307,
      "context_increment_assumed_seconds": 0.08,
      "llm_seconds_expected": 1.3542723127035834
    },
    {
      "k": 4,
      "input": 4400,
      "output": 350,
      "attempts_expected": 1.11,
      "reported_ttft_seconds": 0.44,
      "

## Architectural change and step-function economics
Deterministic routing, concurrent AQ/weather, local HI/KB and one 2500-input/120-output final-generation call. Preserve user language/script; clarify ambiguous slots. Max of two API medians estimates the parallel delay. Resident time is a conservative assumption; memory slots do not prove compute throughput.

In [5]:
remote=max(per["weather"],per["air_quality_observed"]);local=per["heat_index"]+per["knowledge_base"]
changed={}
for m in CANDIDATES:
    seconds=remote+local+1.11*(max(m["ttft_seconds"],120/m["output_tokens_per_second"])+.00005*500)
    unit=(2775*m["input_usd_per_million"]+133.2*m["output_usd_per_million"])/1e6
    changed[m["model"]]={"seconds":seconds,"unit_cost":unit,"monthly_cost":unit*x["monthly_queries"]}
print(json.dumps(changed,indent=2))
feasible=[k for k,v in changed.items() if v["seconds"]<=x["T"]]
chosen=min(feasible,key=lambda k:changed[k]["unit_cost"]) if feasible else min(changed,key=lambda k:changed[k]["seconds"])
holding=math.ceil(max(v["seconds"] for v in changed.values()))
span=x["memory"]["concurrent_sequences_per_gpu"]/((5*.35/120)*holding/60)
lease=720*1.59;api_slope=54*changed[chosen]["unit_cost"]
print("Chosen:",chosen,"Resident seconds:",holding,"Daily Q per GPU step:",span)
print("API slope:",api_slope,"GPU lower-bound slope:",lease/span)
print("No crossing at any positive Q:",lease/span>api_slope)
print("GPU(Q)=1144.8*ceil(Q/28800); API(Q)=",api_slope,"*Q")

{
  "Gemini 2.5 Flash-Lite": {
    "seconds": 2.3746909999941477,
    "unit_cost": 0.00033078,
    "monthly_cost": 27.686286
  },
  "Amazon Nova Lite": {
    "seconds": 3.7288909999941477,
    "unit_cost": 0.00019846799999999998,
    "monthly_cost": 16.611771599999997
  }
}
Chosen: Amazon Nova Lite Resident seconds: 4 Daily Q per GPU step: 28800.0
API slope: 0.010717272 GPU lower-bound slope: 0.03975
No crossing at any positive Q: True
GPU(Q)=1144.8*ceil(Q/28800); API(Q)= 0.010717272 *Q


## Freshness guard and officer summary
The real station observations are historical. Their timings remain valid. The deterministic guard withholds current pollutant numbers when missing, invalid, future or older than 48 hours. The 48-hour cutoff is a conservative application policy, not an official medical threshold.

In [6]:
def observed_status(values,now=None,max_age_hours=48):
    now=now or datetime.now(timezone.utc)
    issues=[];ages={}
    for key in ['pm25','pm10']:
        item=values.get(key)
        if not isinstance(item,dict):issues.append(key+': missing');continue
        value=item.get('value')
        if not isinstance(value,(float,int)) or not math.isfinite(value) or value<0:
            issues.append(key+': invalid value');continue
        try:
            age=(now-datetime.fromisoformat(item['datetime']['utc'].replace('Z','+00:00'))).total_seconds()/3600
            ages[key]=age
            if age<0 or age>max_age_hours:issues.append(key+': stale or future timestamp')
        except (KeyError,ValueError,TypeError):issues.append(key+': invalid timestamp')
    return {'usable_for_current_summary':not issues,'age_hours':ages,'issues':issues,
      'action':'Use readings with original source and observation time' if not issues else 'Withhold current pollution number; state data unavailable/stale; retain provenance for audit'}

raw=list(csv.DictReader((ROOT/"evidence/tool_latency_raw.csv").open(encoding="utf-8")))
real=json.loads([r for r in raw if r["tool"]=="air_quality_observed" and r["success"]=="True"][-1]["detail"])
status=observed_status(real)
assert not status["usable_for_current_summary"]
print(json.dumps(status,indent=2))

{
  "usable_for_current_summary": false,
  "age_hours": {
    "pm25": 17979.522068300834,
    "pm10": 17979.522068300834
  },
  "issues": [
    "pm25: stale or future timestamp",
    "pm10: stale or future timestamp"
  ],
  "action": "Withhold current pollution number; state data unavailable/stale; retain provenance for audit"
}


## Two values not independently verifiable
The actual new-paid TPM entitlement for the chosen routed endpoints and DeepSeek-V3.1 routed cache granularity/numeric lifetime remain unavailable publicly. Consequently no account-capacity certification or DeepSeek cache discount is asserted. This is the uncertainty disclosure explicitly requested in Section 1. The submitted report includes all requested calculations; neither GPU serving tests nor paid LLM tests are required to turn the reported profiles into clearly labelled estimates. No such tests are claimed.